# ARIMA Forecasting — Nile River (Non-Seasonal)
Steps:  ← trend ← seasonality ← stationarity ← ACF/PACF ← ARIMA vs Random Forest ← forecast

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.datasets import nile
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Data


In [ ]:
raw = nile.load_pandas().data
raw["year"] = raw["year"].astype(int)
flow = raw.set_index(pd.to_datetime(raw["year"].astype(str)))["volume"].astype(float)
flow.index.freq = "YS"   # yearly

print("num of years:", len(flow))
print("Missing:", int(flow.isna().sum()))
flow = flow.interpolate()   # if by time or linear
flow.head(100)

## 3. Trend — 5-year moving average
#### does't remove seasonality it just redue the noise by take avg every 5 years

In [ ]:
ma5 = flow.rolling(5, center=True).mean()

plt.figure(figsize=(12,4))
plt.plot(flow, alpha=.6, label="Nile flow")
plt.plot(ma5, linewidth=2, label="5-year MA")
plt.title("Nile Flow + 5-Year Moving Average")
plt.legend(); plt.grid(alpha=.3); plt.show()

### if there was o noise when take avg it will reduce it 

## 4. ACF — auto-correlation
#### trend or level shift ??

In [ ]:
plot_acf(flow, lags=20, zero=False)
plt.title("ACF of Nile flow"); plt.show()

#### look into excel sheet 

## 5. Stationarity — ADF + KPSS
- **ADF**: H0 = non-stationary. لو p < 0.05 ← مستقرة.
- **KPSS**: H0 = stationary (العكس). لو p < 0.05 ← مش مستقرة.
- لو الاتنين اختلفوا ← غالباً فيه level shift (في Nile حوالي 1898).

In [ ]:
adf_p  = adfuller(flow)[1]
kpss_p = kpss(flow, nlags="auto")[1]

print(f"ADF  p = {adf_p:.4f} ->", "stationary" if adf_p < .05 else "non-stationary")
print(f"KPSS p = {kpss_p:.4f} ->", "non-stationary" if kpss_p < .05 else "stationary")

لما الاختبارين يختلفوا، هنجرّب `d=0` و `d=1` الاتنين ونقارن.

## 6. ACF / PACF بعد differencing
PACF بيقطع ← p. ACF بيقطع ← q.

In [ ]:
diff1 = flow.diff().dropna()

fig, ax = plt.subplots(1, 2, figsize=(12,3.5))
plot_acf(diff1,  lags=20, zero=False, ax=ax[0]); ax[0].set_title("ACF (diff) — q")
plot_pacf(diff1, lags=20, zero=False, ax=ax[1], method="ywm"); ax[1].set_title("PACF (diff) — p")
plt.tight_layout(); plt.show()

## 7. Train / Test split (بترتيب الزمن)

In [ ]:
split = int(len(flow) * 0.8)
train, test = flow.iloc[:split], flow.iloc[split:]
h = len(test)
print("Train:", len(train), "| Test:", h)

## 8. models


In [ ]:
orders = {
    "ARIMA(1,0,0)": (1,0,0),
    "ARIMA(1,0,1)": (1,0,1),
    "ARIMA(1,1,0)": (1,1,0),
    "ARIMA(0,1,1)": (0,1,1),
    "ARIMA(1,1,1)": (1,1,1),
    "ARIMA(2,0,1)": (2,0,1),
    "ARIMA(2,1,1)": (2,1,1)
}

preds, aics = {}, {}
for name, order in orders.items():
    fit = ARIMA(train, order=order).fit()
    preds[name] = pd.Series(fit.forecast(h).values, index=test.index)
    aics[name] = fit.aic

# Random Forest: بيتوقع Flow(t) من Flow(t-1), (t-2), (t-3)
LAGS = [1, 2, 3]

def rf_forecast(series, steps):
    df = pd.DataFrame({"y": series})
    for l in LAGS:
        df[f"lag{l}"] = df["y"].shift(l)
    df = df.dropna()
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
    rf.fit(df.drop(columns="y"), df["y"])

    history, out = list(series), []
    for _ in range(steps):
        x = pd.DataFrame([[history[-l] for l in LAGS]], columns=[f"lag{l}" for l in LAGS])
        out.append(rf.predict(x)[0])
        history.append(out[-1])
    return out

preds["Random Forest"] = pd.Series(rf_forecast(train, h), index=test.index)
aics["Random Forest"] = np.nan   # RF مفيهوش AIC

## 9. Evaluation


In [ ]:
rows = []
for name, p in preds.items():
    rows.append({
        "Model": name,
        "MAE":  mean_absolute_error(test, p),
        "RMSE": np.sqrt(mean_squared_error(test, p)),
        "AIC":  aics[name],
    })
metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(metrics.round(1))

plt.figure(figsize=(12,4))
plt.plot(test, "k", linewidth=2, label="Actual")
for name, p in preds.items():
    plt.plot(p, alpha=.7, label=name)
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("Actual vs Forecasts"); plt.show()

## 9b. Rolling one-step forecast (مقارنة أعدل)
التوقع لـ 20 سنة مرة واحدة بيطلع خطوط مستقيمة (الموديل بينسى). هنا بنتوقع **سنة واحدة بس**، وبعدين بنضيف القيمة **الحقيقية** للتاريخ ونتوقع اللي بعدها.
بنضيف كمان **Naive** (توقع إن السنة الجاية = السنة اللي فاتت) كخط أساس: أي موديل لازم يكسبه.

In [ ]:
def rolling_forecast(model_name):
    history = list(train)
    out = []
    for t in range(len(test)):
        if model_name == "Naive":
            out.append(history[-1])
        elif model_name == "Random Forest":
            out.append(rf_forecast(pd.Series(history), 1)[0])
        else:
            fit = ARIMA(history, order=orders[model_name]).fit()
            out.append(fit.forecast(1)[0])
        history.append(test.iloc[t])   # القيمة الحقيقية بتدخل التاريخ
    return pd.Series(out, index=test.index)

roll_preds = {name: rolling_forecast(name)
              for name in ["Naive", "ARIMA(1,0,0)", "ARIMA(1,1,0)", "ARIMA(0,1,1)", "Random Forest"]}

roll_metrics = pd.DataFrame([{
    "Model": n,
    "MAE":  mean_absolute_error(test, p),
    "RMSE": np.sqrt(mean_squared_error(test, p)),
} for n, p in roll_preds.items()]).sort_values("RMSE").reset_index(drop=True)
display(roll_metrics.round(1))

plt.figure(figsize=(12,4))
plt.plot(test, "k", linewidth=2, label="Actual")
for n, p in roll_preds.items():
    plt.plot(p, alpha=.7, label=n)
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("Rolling one-step forecasts"); plt.show()

## the ml model best than naive model a little bit 
#### naive modell = predict the same past value

## 10. Residuals for best model

In [ ]:
best = metrics.loc[0, "Model"]
res = test - preds[best]
print("Best:", best, "| mean:", round(res.mean(),1), "| std:", round(res.std(),1))

fig, ax = plt.subplots(1, 2, figsize=(12,3.5))
ax[0].plot(res); ax[0].axhline(0, ls="--"); ax[0].set_title("Residuals")
plot_acf(res, lags=min(10, len(res)//3), zero=False, ax=ax[1])
plt.tight_layout(); plt.show()

### left plot : how much wrong value between predict and actual
### right plot : how many mistakes correlate to each other , if yes so there was a pattern doesnt catch yet

## 11. Forecast for next 10 years


In [ ]:
future_idx = pd.date_range(flow.index[-1] + pd.offsets.YearBegin(1), periods=10, freq="YS")

if best == "Random Forest":
    final_fc = pd.Series(rf_forecast(flow, 10), index=future_idx)
else:
    fit = ARIMA(flow, order=orders[best]).fit()
    final_fc = pd.Series(fit.forecast(10).values, index=future_idx)

plt.figure(figsize=(12,4))
plt.plot(flow.iloc[-30:], label="History")
plt.plot(final_fc, marker="o", label="10-year forecast")
plt.title(f"Final Forecast — {best}")
plt.legend(); plt.grid(alpha=.3); plt.show()
final_fc.to_frame("Forecast").round(1)